# AI Lab - Practical Question No. 25
## 8-Puzzle Heuristic Evaluator (`h1` Misplaced Tiles & `h2` Manhattan Distance)

- **Syllabus Area**: Informed Search and Heuristics
- **Difficulty**: 🟡 Intermediate
- **Practical Type**: Algorithm Design
- **Core AI Concept**: Heuristic Evaluation for State Spaces
- **Python Concepts Tested**: Matrix iteration, index math (`divmod`)
- **Expected Algorithm / Technique**: Misplaced Tile (`h1`) & Manhattan Distance (`h2`) algorithms
- **Recommended Python Libraries**: None (pure Python standard library)
- **Complexity Expectation**: Time: $\mathcal{O}(N)$, Space: $\mathcal{O}(N)$ where $N = 9$
- **Constraints / Conditions**: Blank space (`0`) must **NOT** be counted in `h1` or `h2`
- **Input Requirements**: Two 3x3 matrices (Current State, Goal State)
- **Expected Output**: Integers for `h1` and `h2`
- **Sample Input**: `Current: [[7,2,4],[5,0,6],[8,3,1]]`, `Goal: [[0,1,2],[3,4,5],[6,7,8]]`
- **Sample Output**: `h1 (Misplaced): 8, h2 (Manhattan): 18`
- **Optional Challenge**: Generate a random solvable board and calculate heuristics
- **University Practical Importance**: Very High

---

### Problem Statement
Given a $3 \times 3$ matrix representing a state of the 8-puzzle (with `0` as the blank space) and a goal state, write a Python program to calculate two heuristics:
1. **$h_1$ (Misplaced Tiles / Hamming Distance)**: Number of non-blank tiles not in their goal position.
2. **$h_2$ (Manhattan Distance / City-Block Distance)**: Sum of orthogonal grid offset distances of all non-blank tiles from their goal positions.

## 1. Theoretical Foundations

For an 8-puzzle state $n$ and goal state $G$ on a $3 \times 3$ grid (indexed $0 \dots 8$ in row-major order):

1. **Coordinate Extraction via `divmod`**:
   For a flattened index $k \in \{0, 1, \dots, 8\}$:
   $$(r, c) = \text{divmod}(k, 3) = \left(\lfloor k / 3 \rfloor,\ k \bmod 3\right)$$

2. **Misplaced Tiles Heuristic ($h_1$)**:
   $$h_1(n) = \sum_{t=1}^{8} \mathbb{1}\big[\text{pos}_n(t) \neq \text{pos}_G(t)\big]$$

3. **Manhattan Distance Heuristic ($h_2$)**:
   $$h_2(n) = \sum_{t=1}^{8} \Big(|r_n(t) - r_G(t)| + |c_n(t) - c_G(t)|\Big)$$

4. **Admissibility & Dominance Hierarchy**:
   $$0 \le h_1(n) \le h_2(n) \le h^*(n) \quad \forall n$$
   Because every misplaced tile contributes $1$ to $h_1$ and at least $1$ to $h_2$, **$h_2$ strictly dominates $h_1$**.

In [ ]:
import sys
from pathlib import Path

# Ensure repository root is on sys.path for local imports
repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.eight_puzzle_heuristics import (
    SAMPLE_CURRENT_STATE,
    SAMPLE_GOAL_STATE,
    CLASSIC_GOAL_STATE,
    ONE_MOVE_STATE,
    UNSOLVABLE_STATE,
    EightPuzzleHeuristicEvaluator,
    misplaced_tiles_heuristic,
    manhattan_distance_heuristic,
    evaluate_heuristics,
    generate_random_solvable_board,
    is_solvable,
    count_inversions,
    demonstrate_blank_space_inadmissibility,
    analyze_dominance_over_samples,
    format_evaluation_report,
    get_eight_puzzle_viva_reference,
)

print("AI Lab 8-Puzzle Heuristic Evaluator module loaded successfully.")

## 2. Canonical Syllabus Sample Verification (Russell & Norvig Fig. 3.28)

- **Current State**: `[[7, 2, 4], [5, 0, 6], [8, 3, 1]]`
- **Goal State**: `[[0, 1, 2], [3, 4, 5], [6, 7, 8]]`
- **Expected Output**: `h1 (Misplaced): 8, h2 (Manhattan): 18`

In [ ]:
current = [[7, 2, 4], [5, 0, 6], [8, 3, 1]]
goal = [[0, 1, 2], [3, 4, 5], [6, 7, 8]]

h1 = misplaced_tiles_heuristic(current, goal)
h2 = manhattan_distance_heuristic(current, goal)
print(f"h1 (Misplaced): {h1}, h2 (Manhattan): {h2}")

result = evaluate_heuristics(current, goal)
print()
print(format_evaluation_report(result))

## 3. Why the Blank Space (`0`) Must NOT Be Counted

Consider a board exactly **1 move** away from the goal (`h* = 1`): `[[1, 0, 2], [3, 4, 5], [6, 7, 8]]`.
If `0` is mistakenly counted as a tile, both `h1` and `h2` report `2 > 1`, violating **admissibility**!

In [ ]:
demo = demonstrate_blank_space_inadmissibility()
print(f"True Optimal Cost h*     : {demo['true_optimal_cost']}")
print(f"Valid (0 excluded)       : h1={demo['h1_valid']} (admissible={demo['h1_valid_admissible']}), "
      f"h2={demo['h2_valid']} (admissible={demo['h2_valid_admissible']})")
print(f"Flawed (0 included)      : h1'={demo['h1_with_blank']} (admissible={demo['h1_with_blank_admissible']}), "
      f"h2'={demo['h2_with_blank']} (admissible={demo['h2_with_blank_admissible']})")

## 4. Optional Challenge: Random Solvable Board Generation & Dominance Study

For any $3 \times 3$ board, reachability is governed by **inversion count parity** (ignoring `0`).
Let us generate a guaranteed-solvable random board, evaluate `h1` and `h2`, and run an empirical dominance study over 100 random solvable boards.

In [ ]:
random_board = generate_random_solvable_board(goal=SAMPLE_GOAL_STATE, seed=42)
rand_res = evaluate_heuristics(random_board, SAMPLE_GOAL_STATE)
print(format_evaluation_report(rand_res))

summary = analyze_dominance_over_samples(num_samples=100, goal=SAMPLE_GOAL_STATE, seed=42)
print(f"\nDominance Study over {summary['num_samples']} Random Solvable Boards:")
print(f"  All Solvable?          : {summary['all_solvable']}")
print(f"  h2 >= h1 on 100%?      : {summary['all_h2_dominates_h1']}")
print(f"  Strict (h2 > h1) Rate  : {summary['strict_dominance_pct']:.1f}%")
print(f"  Mean h1 (Misplaced)    : {summary['mean_h1']:.2f}")
print(f"  Mean h2 (Manhattan)    : {summary['mean_h2']:.2f}")
print(f"  Mean Gap (h2 - h1)     : +{summary['mean_dominance_gap']:.2f}")

## 5. Viva Voce Study Guide

In [ ]:
viva = get_eight_puzzle_viva_reference()
for key, entry in viva.items():
    print(f"=== {key.replace('_', ' ').title()} ===")
    print(f"Q: {entry['question']}")
    print(f"A: {entry['answer']}\n")